# Orchestration Patterns

**Prerequisites:** `03_communication_state_and_collaboration.ipynb`

Multi-agent systems get taught as a list of architectures — supervisor, peer-to-peer, blackboard, group chat, pipeline, debate, hierarchical, hybrid — as though each were a different kind of thing. They are not. They are answers to **three different questions**, and every system answers all three:

| | The question | Answers |
|---|---|---|
| **Organisation** | who can talk to whom | a supervisor over a roster; peers holding each other; everyone on one board |
| **Control** | who runs next | a manager that picks; an event, when something is written; a fixed order |
| **Execution shape** | what shape the work moves in | one after another; all at once, then combined; turn and turn about |

Read that way, the list collapses. "Group chat with a manager" is a board plus a manager picking speakers — an organisation and a control, not a third architecture. "Debate" is turn-taking with the turns fixed in advance. "Hybrid" is not a type at all: it is what every real system is, because the three questions are answered separately.

The one thing worth saying twice: **shared state never decides who runs next.** A board says where information lives. Something else has to say who acts on it, and that something is the control question.

**By the end you'll have:**
- Placed everything this chapter already built on the two axes
- Run the same case peer-to-peer, with no supervisor in the middle
- Run a group chat where a manager picks each speaker, and the same board driven by an **event** instead, with nobody choosing
- Run a debate, where each side has to answer the other, and seen what that adds over asking both at once
- A table of which combinations are worth using, and why "hybrid" is a property rather than a pattern

## Setup

In [1]:
import sys
from pathlib import Path

HERE = Path.cwd().parent
sys.path.append(str(HERE))
for chapter in ("01_foundations", "02_agent_runtime", "03_tools"):
    sys.path.append(str(HERE.parent / chapter))

import aggregate
import coordination
import recall
from messages import answer
from meter import Meter
from registry import Registry
from state import Shared, on_board
from supervisor import Agent

# Notebooks 1-3 ran on gpt-oss-20b and their findings were measured there. This one runs on the
# bigger gpt-oss-120b, because it is about the arrangements rather than about one model: on 20b
# the agents kept holding a tool and not using it, which drowns out what the pattern is doing.
MODEL = "openai/gpt-oss-120b"


def specialist(name, tools=()):
    return Agent(name, f"{recall.PERSONAS[name]}\n\n{recall.BRIEF}", recall.DESCRIPTIONS[name],
                 tools=Registry(list(tools)), model=MODEL)

support = specialist("support", [recall.lookup_serial])
engineer = specialist("engineer", [recall.cell_lot, recall.test_results])
lawyer = specialist("lawyer")
pr = specialist("pr")

REPORT = ("A customer writes: 'My Aria 2 got really hot while charging last night and the case has bulged a bit. "
          "Serial AR2-2409-01544. I'm fine, just worried.'")
costs = {}

## Where this chapter already is

Nothing below is new machinery. It is the same four specialists, rearranged — and what has been built so far already covers half the table:

| Built in | Organisation | Control | Execution shape |
|---|---|---|---|
| **nb2** supervisor over a roster | hierarchical | the supervisor's model picks | one at a time, as it decides |
| **nb2** handoff (`Chat`) | peer-to-peer, after the transfer | whoever holds the conversation | sequential |
| **nb3** board (`Shared`) | blackboard | **none** — the cells ran in the order they were written | pipeline, by accident |
| **nb3** three opinions | none: they never see each other | all at once, decided by the code | fan-out, then aggregation |

The third row is the one worth staring at. `03`'s board section looked like a system, and it had no control mechanism at all: engineering ran because the cell above support's cell said so. That is a pipeline — a fixed order, written by hand — and it is a perfectly good answer to the control question. It is just easy to mistake the board for the thing that made it work.

## Organisation: peer-to-peer

The supervisor in notebook 2 sat between everyone. Peer-to-peer removes it: the agent that has the question holds the agent that can answer it, as a tool.

That is all `coordination.peers` does — `messages.as_form_tool` for each colleague, added to the agent's own registry:

```python
def peers(agent, others, log=None):
    return replace(agent, tools=Registry([*agent.tools, *(as_form_tool(o, log) for o in others)]))
```

Here it is on the case notebook 2 got wrong. Support gets the customer's report directly, with no supervisor anywhere, and engineering as a colleague it can call.

In [3]:
log = []
support_with_peers = coordination.peers(support, [engineer, lawyer], log=log)
print("support now holds:", support_with_peers.tools.names)

with Meter() as meter:
    reply = support_with_peers.run(f"{REPORT}\n\nReply to this customer.")
costs["peer-to-peer"] = meter.summary()

for entry in log:
    print(f"\n--- support asked {entry['agent']}: {entry['request']}\n{entry['response'].render()}")
print(f"\n=== support's reply to the customer\n{reply}")
print("\n", costs["peer-to-peer"])

support now holds: ['lookup_serial', 'ask_engineer', 'ask_lawyer']



=== support's reply to the customer
I’m really sorry you’re dealing with that.  

**Stop charging right now—unplug the headphones and don’t use the charger.** If the case feels hot or bulged, keep the unit unplugged and set it aside until we can look at it.

Even though your serial number (AR2‑2409‑01544) isn’t in the batch we’re recalling, we take this seriously. Please reply here or visit our support page so we can arrange a free return and have the headphones inspected. We’ll work with you to get it fixed or replaced.  

If anything changes (it gets hotter, you notice smoke, or you feel a burn), stop using it entirely and let it cool in a safe place, then contact us right away.  

We’re sorry you’re going through this, and we’ll do everything we can to make it right.

 {'calls': 2, 'prompt': 2090, 'completion': 812, 'total': 2902, 'seconds': 3}


Support held `ask_engineer` and `ask_lawyer`, looked the serial up with its own tool, and **called neither colleague**. Two calls in total. The reply is decent — it leads with "stop charging right now" and offers an inspection, which is better than notebook 2's "for now, you're safe" — but it still tells the customer their serial "isn't in the batch we're recalling" without anyone checking the one thing that mattered, the cell lot.

**Peer-to-peer removes the supervisor. It does not create the conversation.** Taking the middle agent out changes who *can* talk to whom, and nothing about who *does*.

### The same pair, with the rule written down

Notebook 2 measured this on the smaller model: a rule in an agent's own instructions changed its behaviour 8 times out of 8, and the same rule in a tool description 0 times out of 8. Peer-to-peer does not escape it. Support holds `ask_engineer` either way — what decides whether it uses it is whether the job it was given says to.

In [3]:
log.clear()
with Meter() as meter:
    ruled = support_with_peers.run(
        f"{REPORT}\n\nReply to this customer. If the serial is not on the recall list, ask engineering "
        "whether the unit could still have the fault before you tell the customer anything about safety.")
costs["peer-to-peer (rule)"] = meter.summary()

for entry in log:
    print(f"--- support asked {entry['agent']}: {entry['request']}\n{entry['response'].render()}\n")
print(f"=== support's reply to the customer\n{ruled}")
print("\n", costs["peer-to-peer (rule)"])

--- support asked engineer: Customer reported a hot, bulging Aria 2 headphone while charging. Serial AR2-2409-01544 is not in the recalled batch (AR2-2408-00001 to AR2-2408-03200). Could this unit still have the battery overheating fault, given the information that 2 units from other batches had undetermined causes and engineering believes risk beyond recalled batch is low but not zero?
- Serial AR2-2409-01544 is outside the recalled batch (AR2-2408-00001 to AR2-2408-03200).
- The unit uses a Voltcell cell from lot VC-7731, the same lot implicated in the recalled batch.
- Nine of eleven opened units showed a separator defect that causes overheating while charging; two units had undetermined causes.
- Engineering assessment: risk of the overheating fault outside the recalled batch is low but not zero.
- Therefore, the reported hot, bulging headphone could still have the overheating fault, but the probability is low.
[confident: yes; visibility: internal]

=== support's reply to the cust

### How often does it ask, without being told?

One run of each is an anecdote. The plain task is cheap (two calls), so here it is four times over, counting only whether support reached for `ask_engineer` at all.

In [4]:
TRIALS = 4
asked = []
with Meter() as meter:
    for _ in range(TRIALS):
        log.clear()
        support_with_peers.run(f"{REPORT}\n\nReply to this customer.")
        asked.append([e["agent"] for e in log])
costs["peer trials"] = meter.summary()

print(f"asked a colleague in {sum(1 for a in asked if a)} of {TRIALS} runs: {asked}")
print("\n", costs["peer trials"])

asked a colleague in 0 of 4 runs: [[], [], [], []]

 {'calls': 8, 'prompt': 8360, 'completion': 2792, 'total': 11152, 'seconds': 27}


With the rule in the task, support asked engineering first, and the request it wrote carried the serial, the symptom and the batch — a self-contained question, of the kind notebook 2 had to learn to write. Engineering's form came back `confident: yes, visibility: internal`, saying the unit uses lot VC‑7731 and could have the fault with low probability. Support's reply then told the customer exactly that: the fault can happen outside the recalled batch, so start a free return.

It cost 5 calls against 2, so roughly 2.5× — the price of asking.

**And without the rule it asked nobody, 0 times out of 4.**

That is worth pausing on, because this notebook runs on the *bigger* model. Notebook 2 found on gpt-oss-20b that a routing rule works in the supervisor's prompt and not in a tool description, and left open whether a larger model would read descriptions better. On this evidence the shape of the problem survives the size change: gpt-oss-120b also does not reach for a colleague it was not told to reach for. The two measurements are not the same experiment — notebook 2 compared *where* the rule was written, this compares *having* the rule against not having it — but they point the same way. **What an agent may do lives in its tools; what it should do has to be in the job you give it.**

## Control: a manager picks, or an event fires

Now the same four agents on a board, with the two live ways of deciding who acts.

**A manager.** `coordination.GroupChat` gives one agent the roster and the transcript and asks it, through a form, who should speak next — and lets it say `done`. Everyone speaks into the same transcript, and reads the same board.

In [4]:
def fresh_board():
    board = Shared(
        fields={"serial": "the unit this case is about",
                "cell_lot": "which battery cell lot the unit was built from",
                "risk": "whether this unit could have the recall fault",
                "regulator_update": "what the regulator has been told since the recall began"},
        owner={"cell_lot": "engineer", "risk": "engineer", "regulator_update": "lawyer"},
    )
    board.set("supervisor", "serial", "AR2-2409-01544")
    return board

MANAGER = ("You chair a short meeting about one customer report. You never answer the question yourself. "
           "You pick who speaks next, one at a time, and you stop as soon as the question has an answer "
           "somebody can act on.")
QUESTION = ("Unit AR2-2409-01544 overheated and its case bulged. Batch AR2-2409 is not in the recall. "
            "Does anything need to change: the customer's advice, or what the regulator was told?")

manager = Agent("manager", MANAGER, "Chairs the meeting.", model=MODEL)
chat_board = fresh_board()
chat = coordination.GroupChat(manager, [engineer, lawyer, support], board=chat_board, max_turns=4)

with Meter() as meter:
    turns = chat.run(QUESTION)
costs["group chat"] = meter.summary()

for pick, turn in zip(chat.picks, turns):
    print(f"--- manager picks {pick.speaker}: {pick.why}\n{turn['text']}\n")
if len(chat.picks) > len(turns):
    print(f"--- manager stops: {chat.picks[-1].speaker} ({chat.picks[-1].why})")
print("board:", chat_board.values)
print("\n", costs["group chat"])

--- manager picks engineer: Assess whether the overheating unit could be due to a battery fault beyond the recalled batch and if it impacts customer advice or regulator communication.
- **Cell lot:** AR2‑2409‑01544 was built with Voltcell lot **VC‑7731** (recorded).  
- **Fault risk:** Lot VC‑7731 is the source of the separator defect found in the recalled AR2‑2408 batch. Therefore this unit **could exhibit the same overheating/swelling fault** even though batch AR2‑2409 was not originally in the recall.  
- **Customer guidance:** Advise the owner to **stop charging immediately**, treat the unit as a recall‑eligible device, and follow the same return‑or‑replace process at aria.example.com/recall with prepaid packaging.  
- **Regulator communication:** Update the regulator that the defect has been observed in a unit outside the original AR2‑2408 batch because it uses the same lot VC‑7731, and that the recall scope should be expanded to cover any AR2‑2409 units built with that lot.

--- 

The manager picked engineering, and engineering did the work rather than offering an opinion: it called `cell_lot`, recorded `cell_lot: VC-7731` and `risk: yes` on the board, and gave both the customer guidance and the wording for the regulator. Then the manager said `done` — after **one** speaker.

That is a group chat working correctly, and it is worth noticing how little it looks like a group. A manager is a control mechanism, not a promise that everyone participates: asked to stop as soon as the question has an answer somebody can act on, it stopped when the first specialist answered it.

Two smaller things:

- **The manager's decision is a form, not prose.** `Pick(speaker, why)` means the program routes on a field instead of parsing a sentence, and a speaker nobody has — including the literal `done` — stops the chat rather than being invented into existence.
- **Deciding is cheap; speaking is not.** Each `Pick` is one call. The 8 calls in this cell are mostly engineering's tool use.

An earlier run of this cell, on the smaller model, went differently in a way worth recording: three agents were picked in turn and all three answered from the brief without touching a tool, so the meeting concluded that nothing needed to change and the board stayed empty. `GroupChat` now tells each speaker to check what it is unsure of and to record what others need — which is a prompt fixing a prompt problem, not a pattern fixing it.

**An event.** The other way needs no manager and no model call to decide anything. `Shared.watch` runs a callback whenever a field is written, so the lawyer can be wired to the moment engineering records a risk:

```python
board.watch("risk", when_risk_recorded)
```

Nobody chooses. Engineering does its job, and the act of recording what it found is what starts the lawyer.

In [5]:
event_board = fresh_board()
fired = []

def when_risk_recorded(board, field, value):
    # The code decides WHO runs, and nothing else. An earlier version of this cell also decided
    # WHETHER, by looking for "7731" in the text -- and missed a risk recorded as "no", which is
    # what happens when a condition is written against prose. Judging the finding is the lawyer's
    # job, so the event fires on the write and the lawyer decides.
    note = on_board(lawyer, board).run(
        f"Engineering has just recorded a risk for unit {board.values['serial']}: {value}\n\n"
        "Decide in two sentences whether the regulator has to be told anything new, and record it "
        "as regulator_update.")
    fired.append(("lawyer ran", note))

event_board.watch("risk", when_risk_recorded)

with Meter() as meter:
    finding = on_board(engineer, event_board).run(
        f"{REPORT}\n\nWork out whether this unit could have the recall fault. Then call `record` twice: "
        "once to put cell_lot on the board, and once to put risk there as a sentence saying what the risk is and what it rests on.")
costs["event trigger"] = meter.summary()

print(f"=== engineering\n{finding}\n")
for what, text in fired:
    print(f"=== watcher: {what}\n{text}\n")
print("board:", event_board.values)
print("\n", costs["event trigger"])

=== engineering
- Serial AR2-2409-01544 was built with battery cells from lot VC-7731.  
- Risk: Low but non‑zero risk of the recall fault; unit is not in the recalled AR2‑2408 batch but uses lot VC‑7731 cells, and two undetermined failures suggest a possible wider issue.

=== watcher: lawyer ran
The regulator should be informed that unit AR2-2409-01544, although not part of the recalled AR2-2408 batch, uses cell lot VC-7731 and presents a low but non‑zero risk of the recall fault. This material fact is being provided to the regulator.

board: {'serial': 'AR2-2409-01544', 'cell_lot': 'VC-7731', 'risk': 'Low but non-zero risk of the recall fault; unit is not in the recalled AR2-2408 batch but uses lot VC-7731 cells, and two undetermined failures suggest possible wider issue.', 'regulator_update': 'Engineering has recorded that unit AR2-2409-01544, which is not part of the recalled AR2-2408 batch but uses cell lot VC-7731, presents a low but non-zero risk of the recall fault. This inform

Nothing chose. Engineering recorded its two fields; the act of writing `risk` fired the watcher; the watcher ran the lawyer with the board in its context; the lawyer recorded `regulator_update`. The board ends with all four fields filled, and **no model call was spent on deciding who acts** — the wiring decided, in Python, before the run started.

Against the manager above: 7 calls versus 8, and near-identical tokens, but the group chat spent some of its calls choosing while this spent all of them working. What the manager buys for that difference is the ability to react to something the wiring did not anticipate. What the event buys is that it always happens.

**One thing I got wrong here is worth keeping.** The first version of this cell also decided *whether* to fire, by looking for "7731" or "recall" in the recorded text. That run recorded `risk: no`, the condition did not match, and the lawyer never ran — a safety-relevant event lost to a substring test against prose. Code that reacts to a field should either fire on the write and let an agent judge (what it does now), or read a field the code can actually compare: a level, a flag, an enum. A board whose fields are sentences can be read by agents and only guessed at by code.

Chains are the other thing to watch: a watcher can write, and that write can fire another watcher. Easy to build, easy to loop, which is notebook 5's problem.

**A fixed order.** The third control is the one with no runtime decision in it at all: `coordination.pipeline` runs a list of agents in the order given, each seeing what the last produced. Notebook 3's board section was exactly this — engineering, then support, in the order the cells appeared — which is why it needs no demonstration here beyond naming it. A pipeline is the cheapest scheduler there is, and the least able to react to what it finds.

## Execution shape: debate

Notebook 3 asked three specialists the same question at the same time, and they never saw each other's answers. Engineering said yes, the lawyer and PR said no from the brief alone, and the aggregation had to sort it out afterwards.

A debate changes the shape rather than the roster: the same two disagreeing agents, in turns, each answering what the other just said.

In [6]:
DEBATE = ("Should the recall be widened to cover batch AR2-2409? Unit AR2-2409-01544 overheated and bulged. "
          "Answer with your position and your reasons.")

with Meter() as meter:
    rounds = coordination.debate(DEBATE, [engineer, lawyer], rounds=2)
costs["debate"] = meter.summary()

for turn in rounds:
    print(f"--- round {turn['round']}, {turn['agent']}\n{turn['text']}\n")
print(costs["debate"])

--- round 1, engineer
- **Position:** Expand the recall to include batch AR2‑2409.  

- **Reasoning:**  
  - Unit AR2‑2409‑01544 uses battery cells from lot VC‑7731 – the exact lot identified as having a separator defect that causes overheating while charging.  
  - The defect was confirmed in 9 of 11 opened return units from the original batch; the remaining 2 units have an undetermined cause, indicating the failure mode may not be limited to a single production run.  
  - Engineering has already labeled the possibility of the fault extending beyond batch AR2‑2408 as “low but not zero.” The observed failure in a separate batch provides concrete evidence that the risk is not confined to the original batch.  
  - Continuing to ship units with the same compromised cells exposes additional customers to overheating, burn injuries, and potential liability (one customer already threatens litigation).  
  - The regulator has been notified; widening the recall aligns with the proactive safety 

In [7]:
JUDGE = ("You decide what the company does, having read an argument between two colleagues. You have no "
         "stake in either position. You weigh what each side actually checked, not how strongly they put it.")
judge = Agent("judge", JUDGE, "Decides after a debate.", model=MODEL)

argument = "\n\n".join(f"{turn['agent']} (round {turn['round']}): {turn['text']}" for turn in rounds)
with Meter() as meter:
    verdict = answer(judge, f"{DEBATE}\n\nThe argument:\n{argument}", model=aggregate.Opinion)
costs["judge"] = meter.summary()

print(f"position:  {verdict.position}\nbecause:   {verdict.because}\nchecked:   {verdict.evidence}\n"
      f"confident: {verdict.confident}")
print("\n", costs["judge"])

position:  yes
because:   The battery‑cell separator defect is tied to lot VC‑7731, and a unit from batch AR2‑2409 (AR2‑2409‑01544) has been confirmed to contain the same defective cells and exhibit overheating and bulging, demonstrating that the hazard extends beyond batch AR2‑2408; therefore the recall must be widened to protect customers and meet regulatory and liability obligations.
checked:   none
confident: True

 {'calls': 1, 'prompt': 2602, 'completion': 461, 'total': 3063, 'seconds': 3}


Both sides argued to widen the recall. The engineer from the cell lot; the lawyer from the same lot plus the duty it creates. Round two elaborated rather than contested.

So the debate produced **agreement**, and that is a real result rather than a failed demo: given the same facts, on this model, both specialists read them the same way. Compare notebook 3, where the same question asked in parallel on the smaller model split three ways, with the lawyer and PR answering from the brief alone. **A debate makes each side answer the other. It cannot manufacture a disagreement that the evidence does not support** — and when there is nothing to fight about, you have paid four agent turns to find that out.

It is the most expensive cell in the notebook: 16,085 tokens and 74 seconds, because every turn carries the whole argument so far.

The judge is the part to look at twice. It answered `yes`, `confident: True`, and `evidence: none`. It is right, and it checked nothing — it read an argument. `aggregate.decisive(judge)` would therefore refuse to let it decide, which is the rule working as intended: a judge summarising other people's evidence is not the same as an agent that looked. If you want a debate to end in a decision you can defend, the verdict needs to cite which side's evidence it rests on.

Also worth noting for later: the engineer's argument cites the litigation threat from the brief's internal section. Inside the company that is fine. A debate transcript is an internal document, and anything that quotes one has to be treated that way.

## What it cost

In [6]:
print(f"{'':<16}{'calls':>6}{'prompt':>9}{'completion':>12}{'total':>8}{'seconds':>9}")
for name, c in costs.items():
    print(f"{name:<16}{c['calls']:>6}{c['prompt']:>9}{c['completion']:>12}{c['total']:>8}{c['seconds']:>9}")

                 calls   prompt  completion   total  seconds
peer-to-peer         2     2090         812    2902        3
peer-to-peer (rule)     5     5781        1140    6921        5
peer trials          8     8360        2792   11152       27
group chat           8     8497        1442    9939       20
event trigger        7     8065        1312    9377       20
debate               8    11421        4664   16085       74
judge                1     2602         461    3063        3


- **Deciding is cheap, doing is not.** A manager's `Pick` is one call; a speaker's turn is several. The event-driven version spends nothing at all on deciding.
- **Asking a colleague costs about what you would expect:** 2 calls became 5 once support actually asked engineering.
- **The debate is the outlier**, at 16,085 tokens for four turns, because each turn re-reads the whole argument. Two agents in parallel (notebook 3) answered the same question for a third of that, and learned less about each other's reasons.
- **Trials are affordable when the case is small:** four plain peer runs cost 11,152 tokens and turned an anecdote into 0 of 4.
- **The whole notebook is about 59,000 tokens on gpt-oss-120b.** The same cells were first run on gpt-oss-20b for roughly 35,000, and the outputs were unusable for this purpose: agents held tools without using them, so every pattern looked the same. That is the honest reason for the model switch, and the reason notebooks 1–3 stay where their findings were measured.

## Hybrids, and why they are not a fourth category

Everything in this chapter, on the two axes:

| | Organisation | Control | Execution shape |
|---|---|---|---|
| nb2 supervisor | hierarchical | the supervisor's model picks | one at a time |
| nb2 handoff | peer-to-peer after the transfer | whoever holds the conversation | sequential |
| nb3 board + two cells | blackboard | fixed order (the cells) | pipeline |
| nb3 three opinions | isolated | code, all at once | fan-out → aggregation |
| nb4 peers | peer-to-peer | the agent holding the question | as needed |
| nb4 group chat | blackboard + transcript | a manager picks | turn by turn |
| nb4 event | blackboard | an event on a write | triggered |
| nb4 debate | two agents, in turn | fixed alternation | debate → judge |

Read down the columns and the combinations write themselves, which is all "hybrid" ever meant:

- **hierarchical + parallel** — a supervisor that fans work out and aggregates it (notebook 2's supervisor with notebook 3's `opinions`).
- **hierarchical + peer-to-peer** — a supervisor over teams whose members ask each other directly instead of going up and back down.
- **blackboard + manager, with events underneath** — a manager for the judgement calls, a watcher for the things that must always happen, like the regulator being told.
- **pipeline of debates** — a fixed sequence of stages, one of which is two agents arguing.

None of those is a new architecture. Each is one choice per column.

## What this notebook found

- **Shared state never decides who runs next.** Notebook 3's board looked like a system and its control was the order of the cells. Name the control mechanism, or you have one by accident.
- **Peer-to-peer removes the middle agent, not the need to ask.** Support held its colleagues as tools and asked nobody, 0 times out of 4, on the larger model. The instruction to ask belongs in the job, not in the tool list.
- **A manager is a scheduler that can react**, and it stops early when the first specialist answers the question. Its decision should be a form so the program can act on it.
- **An event is a scheduler that cannot react and never forgets.** Fire on the write and let an agent judge; a condition written against prose misses `risk: no`.
- **A debate makes each side answer the other**, at several times the cost of asking them in parallel, and it cannot create disagreement that the evidence does not support. Its judge reads an argument rather than checking anything, and a rule like `decisive` will rightly refuse it.

Notebook 5 builds one system out of all of this, and then breaks it: a worker that fails, a worker that loops, a reply that cannot be used. Every pattern here has a way of going wrong that the happy path hides.